In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/jigyasadoharey/tfidf-model/__results__.html
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/submission.csv
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/__notebook__.ipynb
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/__output__.json
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/tfidf.pkl
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/custom.css
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/wandb/run-20260625_183253-pjniy5yp/run-pjniy5yp.wandb
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/wandb/run-20260625_183253-pjniy5yp/logs/debug.log
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/wandb/run-20260625_183253-pjniy5yp/logs/debug-internal.log
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/wandb/run-20260625_183253-pjniy5yp/files/wandb-summary.json
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/wandb/run-20260625_183253-pjniy5yp/files/config.yaml
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/wandb/run-20260625_18325

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("new_token")


In [3]:
!git clone https://{secret_value_0}@github.com/24f2001460/dl-24f2001460-notebook-t22026.git /kaggle/working/repo
%cd /kaggle/working/repo

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 45 (delta 16), reused 32 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 26.14 KiB | 6.54 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/kaggle/working/repo


In [4]:
!cat src/tfidf_inference.py

import argparse
import joblib
import pandas as pd

from src.tfidf_preprocess import OPTIONS


def load_artefacts(model_path: str):
    bundle = joblib.load(model_path)
    return bundle['vectorizer'], bundle['model']


def build_test_texts(test_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in test_df.iterrows():
        prompt = row['prompt']
        for opt in OPTIONS:
            text = prompt + ' [SEP] ' + str(row[opt])
            rows.append({'id': row.get('id', _), 'option': opt, 'text': text})
    return pd.DataFrame(rows)


def predict(test_df: pd.DataFrame, vectorizer, model) -> pd.DataFrame:
    expanded = build_test_texts(test_df)
    X_test_tfidf = vectorizer.transform(expanded['text'])

    # Probability of class 1 (correct answer)
    proba = model.predict_proba(X_test_tfidf)[:, 1]
    expanded['score'] = proba

    # Rank options by score per question, return top 3 space-separated
    def rank_options(group):
        ranked = group.sort_values('score', 

In [5]:
# ── 3. Run inference ───────────────────────────────────────────────────────
# Adjust --model_path to wherever Kaggle mounted your saved tfidf.pkl dataset
!python -m src.tfidf_inference\
    --test_path  /kaggle/input/competitions/smart-mcq-solver-challenge/test.csv \
    --model_path  /kaggle/input/notebooks/jigyasadoharey/tfidf-model/tfidf.pkl \
    --output_path /kaggle/working/submission.csv

/kaggle/working/repo/src/tfidf_inference.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = expanded.groupby('id').apply(rank_options).reset_index()
Saved 500 predictions to /kaggle/working/submission.csv
